In [1]:
from scripts.Utils import TempRel_Utils
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding
from scripts.Reader import obtain_dataset, id_token_labels
import datasets
import os

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [2]:
datasets, label_list, label2id, id2label = obtain_dataset("TBDense", "E-T")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1936 [00:00<?, ? examples/s]

Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [3]:
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)
tokenizer.add_tokens(["[ES]","[EE]","[TS]","[TE]"])
model.resize_token_embeddings(len(tokenizer))
utils = TempRel_Utils(tokenizer, label_list)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [4]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/1936 [00:00<?, ? examples/s]

Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir="./results/E-T-TempRel",
    logging_dir="./logs/E-T-TempRel",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    num_train_epochs=15,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [11]:
event_timex_temprel = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

C:\Users\Harry\AppData\Local\Temp\ipykernel_18352\2993082861.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  event_timex_temprel = Trainer(


In [12]:
event_timex_temprel.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1
100,1.451100,1.413048,0.403361,0.403361,0.403361
200,1.399500,1.409361,0.319328,0.319328,0.319328
300,1.351500,1.456027,0.420168,0.420168,0.420168
400,1.299800,1.429706,0.432773,0.432773,0.432773
500,1.269500,1.422414,0.428571,0.428571,0.428571
600,1.246000,1.369238,0.441176,0.441176,0.441176
700,1.183200,1.369698,0.500000,0.500000,0.500000
800,1.116300,1.351868,0.407563,0.407563,0.407563
900,1.055800,1.399956,0.478992,0.478992,0.478992
1000,1.020600,1.410049,0.483193,0.483193,0.483193


TrainOutput(global_step=1815, training_loss=1.0064080924042, metrics={'train_runtime': 609.6994, 'train_samples_per_second': 47.63, 'train_steps_per_second': 2.977, 'total_flos': 7641019460321280.0, 'train_loss': 1.0064080924042, 'epoch': 15.0})

In [1]:
event_timex_temprel.evaluate(datasets["test"])

NameError: name 'event_timex_temprel' is not defined

In [9]:
event_timex_temprel.save_model("./results/E-T-TempRel/final_model")